# Pycytominer Processing — Case Overview

Normalizes, aggregates, and applies feature selection to SingleSlice outputs from `1_FeatureSorting.ipynb` (`FeaturesImages_070426_none`). Results saved to `1_Data/results/sections/`.

---

## Plate Layout

384-well plate divided into 6 equal regions (8×8 wells):

|              | Cols 1–8   | Cols 9–16  | Cols 17–24 |
|--------------|------------|------------|------------|
| **Rows A–H** | Sectant 1  | Sectant 2  | Sectant 3  |
| **Rows I–P** | Sectant 4  | Sectant 5  | Sectant 6  |

---

## Cases

| Case | Plate | Wells | Z-slices | Output file |
|------|-------|-------|----------|-------------|
| Section 1 — single z=2 | `stained-one` | all | z=2 | `selected_section1_z2.parquet` |
| Section 1 — single z=7 | `stained-one` | all | z=7 | `selected_section1_z7.parquet` |
| Section 1 — single z=11 | `stained-one` | all | z=11 | `selected_section1_z11.parquet` |
| Section 1 — 3 planes | `stained-one` | all | 3 evenly spaced | `selected_section1_sparse3.parquet` |
| Section 1 — 6 planes | `stained-one` | all | 6 evenly spaced | `selected_section1_sparse6.parquet` |
| Section 1 — 9 planes | `stained-one` | all | 9 evenly spaced | `selected_section1_sparse9.parquet` |
| Section 1 — 12 planes | `stained-one` | all | z=0–11 | `selected_section1_12planes.parquet` |
| Section 1 — all planes | `stained-one` | all | all | `selected_section1_all.parquet` |
| Section 2 | `stained-two-three` | cols 9–16 | all | `selected_section2.parquet` |
| Section 3 | `stained-two-three` | cols 17–24 | all | `selected_section3.parquet` |
| Section 4 ⚠️ | `stained-four-five` | cols 1–8 | all | `selected_section4.parquet` |
| Section 5 | `stained-four-five` | cols 9–16 | all | `selected_section5.parquet` |
| Section 6 — 10X | `stained-six` | all | all | `selected_section6_10x.parquet` |
| Section 6 — 40X | `stained-six-v2` | all | all | `selected_section6_40x.parquet` |

**Notes:**
- Section 1 currently has only 1 well (E08) — re-run `1_FeatureSorting.ipynb` with `cp_id=11637` to get the full plate.
- Section 4 is **uncleared** — segmentation and feature extraction may be unreliable. Treat results with caution.


In [ ]:
# --- repo path bootstrap ---
import sys, pathlib
ROOT = next(p for p in pathlib.Path.cwd().parents if (p / "utils" / "paths.py").is_file())
sys.path.insert(0, str(ROOT))
from utils.paths import (profiles, features, feature_output, figdir, metadata,
                         data_dir, derived, require)
from utils.panels import save_panel

import pandas as pd
import numpy as np
import os

# Pycytominer
from pycytominer import feature_select
from pycytominer import normalize
# from pycytominer import aggregate

# Set current working directory


In [ ]:
def list_features(df):
    # List features
    list_of_selected_features = list(df.columns.values)
    list_of_metadata = list(df.columns[df.columns.str.contains("Metadata_")])
    list_of_selected_features = list(set(list_of_selected_features) - set(list_of_metadata))
    
    return list_of_selected_features, list_of_metadata

In [ ]:
cell_line  = 'HCT116'
data_dir   = str(features("exp2_spheroid_size", "SingleSlice")) + "/"
output_dir = derived("exp2_spheroid_size", "sections")   # derived/, never the deposit
os.makedirs(output_dir, exist_ok=True)


In [ ]:
files = [f for f in os.listdir(data_dir) if cell_line in f and 'MedianAgg' in f]
data = pd.concat([pd.read_parquet(data_dir + f) for f in files], ignore_index=True)
data['_col'] = data['Metadata_Well'].str[1:].astype(int)
print(f'Loaded {data.shape[0]} wells x {data.shape[1]} columns')
print('Plates:', data['Metadata_Barcode'].unique().tolist())
print('Z-slices available:', sorted(data['Metadata_z'].unique().tolist()))


In [ ]:
def process_case(data, case_name, barcodes=None, col_range=None, z_slices=None):
    df = data.copy()
    if barcodes is not None:
        df = df[df['Metadata_Barcode'].isin(barcodes)]
    if col_range is not None:
        df = df[(df['_col'] >= col_range[0]) & (df['_col'] <= col_range[1])]
    if z_slices is not None:
        df = df[df['Metadata_z'].isin(z_slices)]
    if df.empty:
        print(f'[{case_name}] No data after filtering — skipping.')
        return None
    print(f'[{case_name}] {df["Metadata_Barcode"].nunique()} plate(s), '
          f'{df["Metadata_z"].nunique()} z-slice(s), {df["_col"].nunique()} well-cols')

    # Normalize per plate x z-slice
    df['Metadata_plate_slice'] = df['Metadata_Barcode'] + '_' + df['Metadata_z'].astype(str)
    features_list = list_features(df)[0]
    normalized_parts = []
    for unit in df['Metadata_plate_slice'].unique():
        temp = df[df['Metadata_plate_slice'] == unit]
        if temp[temp['Metadata_cmpdname'] == 'dmso'].empty:
            print(f'  [{case_name}] No DMSO in {unit} — skipping slice.')
            continue
        norm_temp = normalize(temp, features=features_list, image_features=False,
                              meta_features='infer',
                              samples="Metadata_cmpdname == 'dmso'",
                              method='standardize')
        normalized_parts.append(norm_temp)
    if not normalized_parts:
        print(f'[{case_name}] No slices could be normalized — skipping.')
        return None
    normalized = pd.concat(normalized_parts, ignore_index=True)

    # Aggregate across z-slices (median per well)
    features_list = list_features(normalized)[0]
    drop_from_meta = ['Metadata_z', 'Metadata_PlateWell', 'Metadata_plate_slice', '_col']
    meta_cols = [c for c in normalized.columns if c not in features_list and c not in drop_from_meta]
    aggregated = normalized.groupby(['Metadata_PlateWell']).agg(
        {**{c: 'first'  for c in meta_cols},
         **{c: 'median' for c in features_list}}
    ).reset_index()

    # Feature selection + clipping
    to_clip = feature_select(aggregated, features=list_features(aggregated)[0],
                             operation=['variance_threshold', 'correlation_threshold', 'drop_na_columns'])
    selected = pd.concat([
        to_clip[list_features(to_clip)[1]],
        to_clip[list_features(to_clip)[0]].clip(lower=-40, upper=40, axis=1)
    ], axis=1)

    # output_dir comes from profiles(..., "<dir>/"), which is a Path -- and Path drops the
    # trailing slash, so string-formatting it glued the directory name onto the
    # filename and wrote the table *beside* the folder instead of inside it.
    out_path = output_dir / f'selected_{case_name}.parquet'
    selected.to_parquet(out_path)
    print(f'  -> saved {selected.shape[0]} wells x {selected.shape[1]} cols to {out_path}')
    return selected


## Section 1 — `colo8-sectant-P2-stained-one`

In [ ]:
s1_barcode = ['colo8-sectant-P2-stained-one']
z_all = sorted(data[data['Metadata_Barcode'].isin(s1_barcode)]['Metadata_z'].unique().tolist())
print('Available z-slices for section 1:', z_all)
print('Total slices:', len(z_all))

# Acquisition parameters
dz_orig = 5.0   # µm between original planes
dz_new  = 1.5   # µm between new planes

# 12 comparable planes: match original 5µm spacing in new 1.5µm data
z_12planes = [0, 3, 7, 10, 13, 17, 20, 23, 27, 30, 33, 37]

# Single planes: original plane index converted to new z
z_orig2new = {2: 7, 7: 23, 11: 37}  # orig plane -> new slice

# Sparse: evenly spaced across full z range
z_sparse = {3: [0, 24, 48], 6: [0, 10, 19, 29, 38, 48], 9: [0, 6, 12, 18, 24, 30, 36, 42, 48]}

print('12 comparable planes:', z_12planes)
print('Single planes (orig->new):', z_orig2new)
print('Sparse selections:', z_sparse)


In [ ]:
# Single planes — at positions equivalent to original z=2, z=7, z=11
for orig_z, new_z in z_orig2new.items():
    process_case(data, f'section1_z{orig_z}', barcodes=s1_barcode, z_slices=[new_z])


In [ ]:
# Sparse sampling: 3, 6, 9 slices evenly spaced across full z range
for n, z_sel in z_sparse.items():
    print(f'n={n}: {z_sel}')
    process_case(data, f'section1_sparse{n}', barcodes=s1_barcode, z_slices=z_sel)


In [ ]:
# 12 planes matching original 5µm acquisition spacing (new data at 1.5µm)
process_case(data, 'section1_12planes', barcodes=s1_barcode, z_slices=z_12planes)


In [ ]:
# All available planes
process_case(data, 'section1_all', barcodes=s1_barcode)


## Sections 2 & 3 — `colo8-sectant-P2-stained-two-three`

In [ ]:
s23_barcode = ['colo8-sectant-P2-stained-two-three']
process_case(data, 'section2', barcodes=s23_barcode, col_range=(9,  16))
process_case(data, 'section3', barcodes=s23_barcode, col_range=(17, 24))


## Sections 4 & 5 — `colo8-sectant-P2-stained-four-five`

In [ ]:
s45_barcode = ['colo8-sectant-P2-stained-four-five']
process_case(data, 'section4', barcodes=s45_barcode, col_range=(1, 8))
process_case(data, 'section5', barcodes=s45_barcode, col_range=(9, 16))


## Section 6 — 10X vs 40X

In [ ]:
process_case(data, 'section6_10x', barcodes=['colo8-sectant-P2-stained-six'])
process_case(data, 'section6_40x', barcodes=['colo8-sectant-P2-stained-six-v2'])
